Running this notebook reproduces my LB 14.49721 submission from the dataset [mrmeows/qcc-1375-open-circuits](https://www.kaggle.com/datasets/mrmeows/qcc-1375-open-circuits) (Apache-2.0). Every file is checked against a pinned SHA-256.

# QCC: open circuits, methods and negative results (public LB 14.497)

## Motivation

With three months remaining, I would rather the leaderboard be decided by new circuit architectures than by who
assembles the best public components first. This release contains my current best circuits, the generators and
audits behind them, the approaches that failed (with measurements), and the bounds I found. Everything is Apache-2.0.

## Summary

| Task | Circuit | Q | Mean Toffolis | Cost | Ratio |
|---|---|---:|---:|---:|---:|
| invsqrt | 68-qubit structural build | 68 | 1,448.1 | 120,219.30 | 17.27 |
| dqi | Gluzdov round12 (public, unchanged) | 881 | 964,394 | 8.917e11 | 27.41 |
| dfthcsos | 8-bit, R=4, B=303, C=128 | 448 | 23,500 | 5.214e8 | 6.435 |

Public LB score: 14.49721. All costs come from the unmodified official verifiers at full default shot counts,
on the exact bytes in the dataset; file hashes are in `circuits/SHA256SUMS`.

The cost functions reward different trades. For dqi (R·Q²·T), one qubit near Q=880 is worth about 0.23%.
For invsqrt (Q·C), one qubit at Q=68 is worth about 1.5%, roughly 22 Toffoli-equivalents.
For dfthcsos (λ_eff·T·Q), the largest gain came from a discontinuity in Q.

## dfthcsos (4.24 → 6.44)

**Exact lookups on a fixed factorization.**
- The ANGLE lookups were the dominant term in T (about 60% at the start).
- The valid address set {0,…,875} is closed under bit subsets. Each output bit is therefore an XOR of
  address-bit monomials: 865 Toffolis per circuit, with products erased by X-basis measurement and a phase correction.
- Four-address loaders reduce the persistent angle register from 110 to 30 qubits.
- Partitioned, nested and shared-product lookup compilation then reduced T to 17,063.
- None of this changes `submission.npz`, so both accuracy metrics are unaffected (best 10-bit result: 5.62).

**A bound on refitting.**
- Column rotations and isometries of each block preserve the Gram matrix, the one-body shift and E_gap; only Λ changes.
- From `Λ₂ ≥ ¼ Σ_ij |(AAᵀ)_ij|`, the attainable improvement in λ_eff on that factorization is below 0.3%.
- Further gains must come from the circuits or from the factorization itself.

**Eight-bit words.**
- In the official cost model, the state-preparation workspace depends on word width. At 8 bits it drops from 607 to
  about 300 qubits, taking total Q from 742 to 448. This does not happen at 9 bits.
- Direct 8-bit rounding of good 10-bit fits gives error₂ ≈ 111 mHa.
- Recovering accuracy required:
  - constrained discrete refits of the angle words with coefficient transport;
  - a larger dictionary (B ≈ 300–350);
  - accepting λ_eff ≈ 47.5–49.5, which the halved Q more than pays for.
- Results: B=333 → 6.15, then B=303 → 6.435.

**Negative results.**
- 7-bit words: error₂ ≈ 60–70 mHa.
- Mixed word widths are not expressible in the submission format.
- Paired (cubature) rounding fixes accuracy, but the doubled dictionary costs more than it saves.
- A B=191/C=136 shape that crosses the 32-step boundary in B+1: continuous error₂ 19.35 mHa, rounded 26.8 mHa.
- Joint diagonalisation and shared bases across non-commuting blocks: invalid.
- The single-group R=1/B=496/C=384 factorization: about 4.15 at 10 bits.

**Open directions.**
- A compact 8-bit fit with fewer vectors stalls near 41 mHa under local moves. Lattice rounding on the
  linearised sensitivity (LLL/Babai) and annealed quantization-aware training are the obvious next steps.
- At Q=448, 7× needs T ≈ 21,500.

## invsqrt (15.04 → 17.27)

The base design is the public lineage (Gluzdov → DePie EXP-222/225a/226a → busyaprime, 78 qubits):
- one-bit normalisation;
- a quadratic per segment from a QROM, with signed-digit Horner rows;
- a grouped curvature lookup erased by measurement;
- a fused phase stage with 44 calibrated rotations.

My changes alter storage and uncomputation only. The polynomial and all rotation angles are unchanged, and each
identity is checked on all 3,133,452 legal inputs:
1. QROM outputs that are Boolean functions of other outputs are recomputed at their point of use.
2. Thirty carry prefixes are completed and erased early (X-basis measurement with a Clifford repair).
3. Increment-tail workspaces are removed by temporarily erasing the top accumulator bit.
4. The second phase pass alternates between two buffers.
5. Selected input bits and normalisation flags are parked in freed slots and restored with one Toffoli each.

Q falls from 78 to 68, and worst error/tolerance stays at 0.99966. `source/invsqrt_68q/` contains the generator
and the exactness and measurement notes.

**Observations.**
- Within this architecture, C appears bounded below by roughly 1,450–1,550; the error budget is 95–99% consumed
  throughout.
- Newton refinement, cubic segments, other table widths, shallower truncation and register recolouring were all
  measured as neutral or worse.
- The reported 20× level (cost ≈ 104k) likely requires a different design.

**Open directions.**
- Exploit the problem's structure: the tolerance is a constant *relative* 2⁻²⁴, the phase is at most one turn, and
  kickback is linear in the output bits, so 1/√x never needs to be stored in full. Computing phase bits directly as
  low-degree functions of x is the most promising avenue I have not completed.
- A 67-qubit variant with a shared curvature block (four outputs from 12 Toffolis) reaches 119,054 (17.44); it is
  not in this release.

## dqi

I use Gluzdov's round12 decoder unchanged: an in-place continued-fraction extraction over the 840-qubit syndrome bank,
with SomeJoint's Bézout/Hankel phase solve, mask projection and history compression. It is not entirely fault-free:
another team observed 2 phase failures in 102,400 shots (R = 1.201 instead of 1.191).

**My own decoder work.**

| Approach | Outcome |
|---|---|
| Berlekamp–Massey and extended-Euclid (streaming, windowed, journaled) | Width lower bounds of 1,704 (BM) and 1,296 (EEA) qubits; not competitive under Q² |
| Low-rank bilinear Forney (64 Toffolis/position) | Impossible: the bilinear rank of GF(64) multiplication is at least 15 |
| Inverse-free magnitude sweep (evaluate M = Ω/σ′ mod σ, zero test, phase) | 45k Toffolis at 851 qubits for the phase stage, against 328k for Chien/Forney |
| Structured Bézoutian preprocessing for M | 871 qubits, 73k Toffolis forward. Correct for degree 35 with regular pivots; deep Euclidean degree drops remain open |

Two identities from the last line:
- In characteristic 2, writing σ(x) = E(x²) + x·O(x²) gives σ′(x) = O(x²). The extended GCD then runs on degree ≤ 17.
- G ← G + (a+1)·F makes G monic without changing the Bézoutian B(F, G), including when a = 0.

**Headroom.** Q cannot go below 840. At 873 qubits and 920k Toffolis the ratio would be about 29; at 850 qubits and
700k Toffolis, about 43. Most of the remaining Toffolis are in the extraction (about 610k, forward and reverse).

## Practical notes

- The official verifiers parallelise with `fork`, so on Windows they run serially. Use Linux, and screen with
  `--shots-per-x 1` before a full invsqrt run.
- kickmix's `flip_if_equal` allocates ancillas inside the C++ layer. Only the built circuit's `num_qubits` is reliable.
- Toffolis inside a measurement-conditioned block are counted only when executed.
- Stripping every Z-basis reset from these circuits leaves the verified scores unchanged, so the resets act on |0⟩.
- A dqi run at 2,048 shots reports R ≈ 7.5 even with zero failures, because of the confidence bound. Compare decoders
  at 102,400 shots.

## Credits

- **invsqrt:** Akmal Xodarev (busyaprime), Phuc Thai (DePie), Dmitrii Gluzdov, marsyas.
- **dqi:** Dmitrii Gluzdov (round12), Agniv Banerjee (SomeJoint), ViberTeam, manwithacat, Phuc Thai.
- **dfthcsos:** Ahmed Berat Özer (group compression), marculera (four-address loaders). Michael Rustad independently
  arrived at the 8-bit, Q=448 regime.
- **Verifiers and kickmix:** Google Quantum AI.

Please keep these credit chains if you build on this.


In [ ]:
import glob, hashlib, pathlib, zipfile
def sha(b): return hashlib.sha256(b).hexdigest()
PIN = {'invsqrt.kmb': '7bde83f48462a245b9307fe0d9db985d9b2e9b5b1901a48b21df96eca2106985',
       'dqi.kmb': '835414f51eff796282be942f86e0f479901597c144481653eed63402cb6e4ea0'}
root = pathlib.Path(next(p for p in glob.glob('/kaggle/input/**/qcc-1375-open-circuits', recursive=True) + glob.glob('/kaggle/input/qcc-1375-open-circuits*')))
files = {}
inv = list(root.rglob('circuits/invsqrt.kmb')) or list(root.rglob('invsqrt.kmb'))
if inv:
    files['invsqrt.kmb'] = inv[0].read_bytes()
    files['dqi.kmb'] = next(root.rglob('dqi.kmb')).read_bytes()
    for p in sorted(root.rglob('circuits/dfthcsos/*')):
        if p.is_file(): files['dfthcsos/' + p.name] = p.read_bytes()
else:
    with zipfile.ZipFile(next(root.rglob('circuits.zip'))) as z:
        for n in z.namelist():
            base = n.split('/')[-1]
            if n.endswith('invsqrt.kmb') or n.endswith('dqi.kmb'): files[base] = z.read(n)
            elif '/dfthcsos/' in '/' + n and base: files['dfthcsos/' + base] = z.read(n)
for k, v in PIN.items(): assert sha(files[k]) == v, k
assert sum(k.startswith('dfthcsos/angle_k_') for k in files) == 15 and 'dfthcsos/submission.npz' in files
with zipfile.ZipFile('/kaggle/working/submission.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for k in sorted(files): z.writestr(k, files[k])
print({k: sha(v)[:12] for k, v in files.items() if not k.startswith('dfthcsos/angle')})
print('wrote submission.zip:', len(files), 'files (LB 14.49721 package)')
